Downloading and verifying the Dataset:

In [ ]:
import kagglehub
dataset_path = kagglehub.dataset_download('sadmanhsakib/hc18-processed-dataset')

import os
print("Dataset stored at: ", dataset_path)
print("Subfolders: ", os.listdir(dataset_path))

Dynamic Data mapping for the data.yaml file:

In [ ]:
import yaml
from pathlib import Path

import cv2
import numpy as np

WORK_DIR = Path("/kaggle/working")
DATASET_ROOT = Path(dataset_path)
KAGGLE_DATASET_DIR = DATASET_ROOT / "yolo"
DATA_YAML = WORK_DIR / "data_patched.yaml"
PROJECT_DIR = WORK_DIR / "fetalmetrics-ai"
TUNE_CFG = WORK_DIR / "runs" / "segment" / "tune" / "best_hyperparameters.yaml"
VAL_IMAGES_DIR = KAGGLE_DATASET_DIR / "images" / "val"
ONNX_PATH = WORK_DIR / "yolov8_hc.onnx"

RUN_BASELINE = "yolov8s_seg_baseline"
RUN_OPTIMIZED = "yolov8s_seg_optimized"
RUN_FINAL = "yolov8s_seg_final"

CLINICAL_AUG = {
    "flipud": 0.0,
    "fliplr": 0.5,
    "degrees": 15.0,
    "scale": 0.15,
    "hsv_h": 0.0,
    "hsv_s": 0.0,
    "hsv_v": 0.15,
}

with open(KAGGLE_DATASET_DIR / "data.yaml", "r") as f:
    config = yaml.safe_load(f)
config["path"] = str(KAGGLE_DATASET_DIR.resolve())
with open(DATA_YAML, "w") as f:
    yaml.safe_dump(config, f)
print(f"Patched data.yaml written to {DATA_YAML}")

def run_weights(run_name: str) -> Path:
    return PROJECT_DIR / run_name / "weights" / "best.pt"

def common_train_kwargs(**overrides):
    kwargs = {
        "data": str(DATA_YAML),
        "epochs": 100,
        "imgsz": 640,
        "batch": 16,
        "device": 0,
        "optimizer": "AdamW",
        "val": True,
        "project": str(PROJECT_DIR),
    }
    kwargs.update(overrides)
    return kwargs

def fastai_mask_path(img_path: Path) -> Path:
    return DATASET_ROOT / "fastai" / "masks" / "val" / img_path.name

def calculate_dice(pred_mask: np.ndarray, gt_mask: np.ndarray) -> float:
    intersection = np.sum((pred_mask > 0) & (gt_mask > 0))
    total_pixels = np.sum(pred_mask > 0) + np.sum(gt_mask > 0)
    if total_pixels == 0:
        return 1.0 if np.sum(gt_mask) == 0 else 0.0
    return (2.0 * intersection) / total_pixels

def evaluate_val_dice(model) -> float:
    dice_scores = []
    for img_path in sorted(VAL_IMAGES_DIR.glob("*.png")):
        mask_path = fastai_mask_path(img_path)
        gt_mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
        if gt_mask is None:
            raise FileNotFoundError(f"Missing ground-truth mask: {mask_path}")

        results = model.predict(str(img_path), imgsz=640, verbose=False)
        pred_mask = np.zeros_like(gt_mask)
        if results[0].masks is not None:
            mask_data = results[0].masks.data.cpu().numpy()[0]
            pred_mask = cv2.resize(
                mask_data,
                (gt_mask.shape[1], gt_mask.shape[0]),
                interpolation=cv2.INTER_NEAREST,
            )
            pred_mask = ((pred_mask > 0.5).astype(np.uint8)) * 255

        dice_scores.append(calculate_dice(pred_mask, gt_mask))

    if not dice_scores:
        raise RuntimeError(f"No validation images found in {VAL_IMAGES_DIR}")
    return float(np.mean(dice_scores))

def select_best_checkpoint():
    from ultralytics import YOLO

    candidates = {
        RUN_BASELINE: run_weights(RUN_BASELINE),
        RUN_OPTIMIZED: run_weights(RUN_OPTIMIZED),
        RUN_FINAL: run_weights(RUN_FINAL),
    }

    print("Comparing validation Dice across all training runs:")
    best_name, best_path, best_dice = None, None, -1.0
    for name, path in candidates.items():
        if not path.exists():
            print(f"  {name}: skipped (checkpoint not found)")
            continue
        dice = evaluate_val_dice(YOLO(str(path)))
        print(f"  {name}: Dice = {dice:.4f}")
        if dice > best_dice:
            best_name, best_path, best_dice = name, path, dice

    if best_path is None:
        raise RuntimeError("No trained checkpoints found to export.")
    print(f"\nSelected for export: {best_name} (Dice = {best_dice:.4f})")
    return best_name, best_path, best_dice

Installing Dependencies:

In [ ]:
!pip install -q ultralytics

Training the segmentation model: Baseline Training

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8s-seg.pt")
model.train(
    **common_train_kwargs(
        name=RUN_BASELINE,
        lr0=0.01,
        patience=10,
        flipud=0.0,
        hsv_h=0.0,
        hsv_s=0.0,
        plots=True,
    )
)

Dice Coefficient Validation (Baseline):

In [ ]:
baseline_model = YOLO(str(run_weights(RUN_BASELINE)))
mean_dice_baseline = evaluate_val_dice(baseline_model)
print(f"Baseline Validation Mean Dice Coefficient: {mean_dice_baseline:.4f}")

Model Optimization with tweaked LR and augmentation settings:

In [ ]:
model = YOLO("yolov8s-seg.pt")
model.train(
    **common_train_kwargs(
        name=RUN_OPTIMIZED,
        lr0=0.005,
        patience=12,
        plots=True,
        **CLINICAL_AUG,
    )
)

Dice Coefficient Validation (Optimized):

In [ ]:
optimized_model = YOLO(str(run_weights(RUN_OPTIMIZED)))
mean_dice_optimized = evaluate_val_dice(optimized_model)
print(f"Optimized Validation Mean Dice Coefficient: {mean_dice_optimized:.4f}")
print(f"Improvement over baseline: {mean_dice_optimized - mean_dice_baseline:+.4f}")

Hyperparameter tuning (from optimized checkpoint): genetic search over LR, weight decay, and augmentation — clinical constraints from `CLINICAL_AUG` are held fixed.

In [ ]:
optimized_ckpt = run_weights(RUN_OPTIMIZED)
if not optimized_ckpt.exists():
    raise FileNotFoundError(f"Run optimized training first: {optimized_ckpt}")

model = YOLO(str(optimized_ckpt))
model.tune(
    data=str(DATA_YAML),
    epochs=30,
    iterations=5,
    optimizer="AdamW",
    plots=False,
    save=False,
    val=True,
    **CLINICAL_AUG,
)

Final training with tuned hyperparameters (continues from optimized checkpoint):

In [ ]:
if not TUNE_CFG.exists():
    raise FileNotFoundError(f"Run hyperparameter tuning first: {TUNE_CFG}")

optimized_ckpt = run_weights(RUN_OPTIMIZED)
if not optimized_ckpt.exists():
    raise FileNotFoundError(f"Run optimized training first: {optimized_ckpt}")

model = YOLO(str(optimized_ckpt))
model.train(
    **common_train_kwargs(
        cfg=str(TUNE_CFG),
        name=RUN_FINAL,
        patience=12,
        plots=True,
        **CLINICAL_AUG,
    )
)

Select the best checkpoint across all runs (by validation Dice) and export to ONNX:

In [ ]:
from ultralytics import YOLO

best_name, best_path, best_dice = select_best_checkpoint()
best_model = YOLO(str(best_path))
onnx_file = best_model.export(format="onnx", simplify=True, imgsz=640)
Path(onnx_file).rename(ONNX_PATH)
print(f"Exported {best_name} (Dice={best_dice:.4f}) to {ONNX_PATH}")